# Compare segmentation candidates

In [ ]:
from __future__ import annotations

import json
import logging
import math
import shutil
import sys
from copy import deepcopy
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List, Tuple

import joblib
import numpy as np
import pandas as pd
import yaml
from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, calinski_harabasz_score, davies_bouldin_score, silhouette_score
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 260)


In [ ]:
# --- Project root + config setup ---
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

print("PROJECT_ROOT =", PROJECT_ROOT)

CONFIG_PATH = PROJECT_ROOT / "configs" / "config.yaml"
print("CONFIG_PATH  =", CONFIG_PATH)

DEFAULT_CONFIG = {
    "paths": {
        "output_dir": "output",
        "log_dir": "output/logs",
    },
    "manifest": {
        "output_subdir": "00_manifest",
    },
    "ssl": {
        "output_subdir": "01_ssl_frames",
    },
    "segmentation": {
        "output_subdir": "02_segments",
    },
    "segment_sweep": {
        "output_subdir": "02b_segment_sweep",
        "selected_refined_subdir": "02_segments_refined",
        "overwrite_candidate_downstream": False,
        "promote_selected_candidate": False,
    },
    "clustering": {
        "segment_vectors_subdir": "03_segment_vectors",
        "kstudy_subdir": "04_kstudy",
        "k_values": [32, 48, 64, 80, 96],
        "random_seeds": [7, 19, 31, 43, 55],
        "batch_size": 256,
        "max_iter": 300,
        "n_init": 10,
        "use_pca": True,
        "pca_components": 128,
        "silhouette_max_samples": 3000,
        "final_model_seed": 31,
        "reassignment_ratio": 0.01,
    },
    "tokens": {
        "token_prefix": "u",
        "token_zero_pad": None,
        "round_time_decimals": 4,
        "include_manifest_metadata_in_jsonl": True,
    },
    "dictionary": {
        "output_subdir": "06_dictionary",
    },
    "audits": {
        "output_subdir": "07_audits",
        "dominant_warning_threshold": 0.80,
        "dominant_high_threshold": 0.90,
        "min_segments_for_flag": 8,
    },
    "comparison": {
        "weights": {
            "segmentation_improvement": 0.30,
            "fragment_control": 0.20,
            "clustering_quality": 0.20,
            "sequence_richness": 0.20,
            "confound_control": 0.10,
        },
        "penalties": {
            "no_single_segment_improvement": 0.10,
            "fragment_explosion": 0.15,
            "confound_worsened": 0.10,
        },
        "fragment_explosion_absolute_pct": 25.0,
        "fragment_explosion_delta_pct": 7.0,
        "confound_worsened_delta_pct": 5.0,
    },
}

def deep_update(base: Dict[str, Any], updates: Dict[str, Any]) -> Dict[str, Any]:
    result = deepcopy(base)
    for key, value in (updates or {}).items():
        if isinstance(value, dict) and isinstance(result.get(key), dict):
            result[key] = deep_update(result[key], value)
        else:
            result[key] = value
    return result

CONFIG = deepcopy(DEFAULT_CONFIG)
if CONFIG_PATH.exists():
    with open(CONFIG_PATH, "r", encoding="utf-8") as f:
        loaded = yaml.safe_load(f) or {}
    CONFIG = deep_update(CONFIG, loaded)

# backward compatibility for uploaded config that still uses singular 'audit'
if "audit" in CONFIG and "audits" not in CONFIG:
    CONFIG["audits"] = deepcopy(DEFAULT_CONFIG["audits"])
    CONFIG["audits"] = deep_update(CONFIG["audits"], CONFIG.get("audit") or {})

def resolve_path(raw_path: str) -> Path:
    path = Path(raw_path)
    if path.is_absolute():
        return path
    return (PROJECT_ROOT / path).resolve()

OUTPUT_DIR = resolve_path(CONFIG["paths"]["output_dir"])
LOG_DIR = resolve_path(CONFIG["paths"]["log_dir"])
LOG_DIR.mkdir(parents=True, exist_ok=True)

MANIFEST_DIR = OUTPUT_DIR / CONFIG["manifest"]["output_subdir"]
SSL_DIR = OUTPUT_DIR / CONFIG["ssl"]["output_subdir"]
BASELINE_SEG_DIR = OUTPUT_DIR / CONFIG["segmentation"]["output_subdir"]
SWEEP_DIR = OUTPUT_DIR / CONFIG["segment_sweep"]["output_subdir"]
REFINED_SEG_DIR = OUTPUT_DIR / CONFIG["segment_sweep"]["selected_refined_subdir"]

SWEEP_PLAN_CSV = SWEEP_DIR / "sweep_plan.csv"
SWEEP_RESULTS_CSV = SWEEP_DIR / "sweep_results.csv"
BASELINE_REFERENCE_JSON = SWEEP_DIR / "baseline_reference_metrics.json"
COMPARISON_SCORECARD_CSV = SWEEP_DIR / "comparison_scorecard.csv"
COMPARISON_REPORT_MD = SWEEP_DIR / "comparison_report.md"
SELECTED_CANDIDATE_MD = SWEEP_DIR / "selected_candidate.md"
SELECTED_CANDIDATE_JSON = SWEEP_DIR / "selected_candidate.json"

print("OUTPUT_DIR       =", OUTPUT_DIR)
print("SWEEP_DIR        =", SWEEP_DIR)
print("REFINED_SEG_DIR  =", REFINED_SEG_DIR)


In [ ]:
# --- Logging setup ---
LOG_FILE = LOG_DIR / "02c_compare_segmentation_candidates.log"

logger = logging.getLogger("stage4_segment_compare")
logger.setLevel(logging.INFO)
logger.handlers.clear()

formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")

file_handler = logging.FileHandler(LOG_FILE, encoding="utf-8")
file_handler.setFormatter(formatter)
logger.addHandler(file_handler)

stream_handler = logging.StreamHandler(sys.stdout)
stream_handler.setFormatter(formatter)
logger.addHandler(stream_handler)

logger.info("Logging initialized")
logger.info("Log file: %s", LOG_FILE)


In [ ]:
# --- Helpers adapted from 03 / 04 / 05 / 06 notebooks ---
def safe_float(value: Any, default: float = 0.0) -> float:
    try:
        if pd.isna(value):
            return default
        return float(value)
    except Exception:
        return default

def safe_int(value: Any, default: int = 0) -> int:
    try:
        if pd.isna(value):
            return default
        return int(value)
    except Exception:
        return default

def safe_div(numer: float, denom: float, default: float = 0.0) -> float:
    denom = float(denom)
    if abs(denom) < 1e-12:
        return default
    return float(numer) / denom

def write_markdown(path: Path, lines: List[str]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))

def load_json(path: Path, default=None):
    if not path.exists():
        return deepcopy(default)
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def sanitize_clip_id(value: str) -> str:
    text = str(value).strip()
    return text.replace("/", "_").replace("\\", "_").replace(" ", "_")

def ensure_candidate_dirs(candidate_dir: Path) -> Dict[str, Path]:
    paths = {
        "root": candidate_dir,
        "segments_csv": candidate_dir / "segments.csv",
        "segment_summary_csv": candidate_dir / "segment_summary.csv",
        "candidate_config_json": candidate_dir / "candidate_config.json",
        "candidate_metrics_json": candidate_dir / "candidate_metrics.json",
        "candidate_eval_metrics_json": candidate_dir / "candidate_eval_metrics.json",
        "candidate_notes_md": candidate_dir / "candidate_notes.md",
        "segment_vectors_dir": candidate_dir / "03_segment_vectors",
        "kstudy_dir": candidate_dir / "04_kstudy",
        "tokens_dir": candidate_dir / "05_tokens",
        "dict_dir": candidate_dir / "06_dictionary",
        "audits_dir": candidate_dir / "07_audits",
    }
    for key in ["root", "segment_vectors_dir", "kstudy_dir", "tokens_dir", "dict_dir", "audits_dir"]:
        paths[key].mkdir(parents=True, exist_ok=True)
    return paths

def load_frame_array(frame_path: Path):
    arr = np.load(frame_path, allow_pickle=False)
    key = "frames"
    if key not in arr.files:
        candidate = [k for k in arr.files if arr[k].ndim == 2]
        if not candidate:
            raise KeyError(f"No 2D frame array found in {frame_path}")
        key = candidate[0]
    frames = arr[key].astype(np.float32, copy=False)
    frame_times = arr["frame_times_sec"] if "frame_times_sec" in arr.files else None
    return frames, frame_times

def aggregate_segment_vector(frames_2d: np.ndarray) -> np.ndarray:
    return frames_2d.mean(axis=0).astype(np.float32)

def build_segment_vectors(segments_df: pd.DataFrame, project_root: Path) -> Dict[str, np.ndarray]:
    rows = []
    frame_cache: Dict[Path, np.ndarray] = {}

    for row in tqdm(segments_df.itertuples(index=False), total=len(segments_df), desc="Building segment vectors"):
        frame_path = Path(row.frame_file)
        if not frame_path.is_absolute():
            frame_path = (project_root / frame_path).resolve()
        if not frame_path.exists():
            alt = (project_root / str(row.frame_file).replace("\\", "/")).resolve()
            if alt.exists():
                frame_path = alt
        if not frame_path.exists():
            raise FileNotFoundError(f"Missing frame file for segment vector build: {frame_path}")

        if frame_path not in frame_cache:
            frames_2d, _ = load_frame_array(frame_path)
            frame_cache[frame_path] = frames_2d
        else:
            frames_2d = frame_cache[frame_path]

        start_i = int(row.start_frame)
        end_i = int(row.end_frame_exclusive)
        if start_i < 0 or end_i > len(frames_2d) or end_i <= start_i:
            raise ValueError(
                f"Invalid frame slice for {row.segment_id}: start={start_i}, end={end_i}, total={len(frames_2d)}"
            )
        seg_frames = frames_2d[start_i:end_i]
        vec = aggregate_segment_vector(seg_frames)
        rows.append(
            (
                str(row.segment_id),
                str(row.clip_id),
                int(row.segment_index),
                float(row.start_sec),
                float(row.end_sec),
                float(row.duration_sec),
                vec,
            )
        )

    if not rows:
        raise ValueError("No segment rows found when building segment vectors.")

    vector_segment_ids = np.array([r[0] for r in rows], dtype="U256")
    vector_clip_ids = np.array([r[1] for r in rows], dtype="U256")
    vector_segment_index = np.array([r[2] for r in rows], dtype=np.int32)
    vector_start_sec = np.array([r[3] for r in rows], dtype=np.float32)
    vector_end_sec = np.array([r[4] for r in rows], dtype=np.float32)
    vector_duration_sec = np.array([r[5] for r in rows], dtype=np.float32)
    vector_matrix = np.stack([r[6] for r in rows]).astype(np.float32)

    return {
        "vectors": vector_matrix,
        "segment_ids": vector_segment_ids,
        "clip_ids": vector_clip_ids,
        "segment_index": vector_segment_index,
        "start_sec": vector_start_sec,
        "end_sec": vector_end_sec,
        "duration_sec": vector_duration_sec,
    }

def run_kmeans_once(X, k, seed, batch_size=256, max_iter=300, n_init=10, reassignment_ratio=0.01):
    model = MiniBatchKMeans(
        n_clusters=k,
        random_state=seed,
        batch_size=batch_size,
        max_iter=max_iter,
        n_init=n_init,
        reassignment_ratio=reassignment_ratio,
    )
    labels = model.fit_predict(X)
    counts = np.bincount(labels, minlength=k)
    return model, labels, counts

def safe_silhouette(X, labels, max_samples=3000, seed=0):
    n_labels = len(np.unique(labels))
    n = len(labels)
    if n_labels < 2 or n_labels >= n:
        return np.nan
    if n > max_samples:
        rng = np.random.default_rng(seed)
        idx = np.sort(rng.choice(n, size=max_samples, replace=False))
        return float(silhouette_score(X[idx], labels[idx]))
    return float(silhouette_score(X, labels))

def safe_calinski(X, labels):
    n_labels = len(np.unique(labels))
    if n_labels < 2 or n_labels >= len(labels):
        return np.nan
    return float(calinski_harabasz_score(X, labels))

def safe_davies(X, labels):
    n_labels = len(np.unique(labels))
    if n_labels < 2 or n_labels >= len(labels):
        return np.nan
    return float(davies_bouldin_score(X, labels))

def pairwise_stability(label_runs):
    if len(label_runs) < 2:
        return np.nan, np.nan
    aris = []
    for i in range(len(label_runs)):
        for j in range(i + 1, len(label_runs)):
            aris.append(adjusted_rand_score(label_runs[i], label_runs[j]))
    return float(np.mean(aris)), float(np.std(aris))

def cluster_size_stats(counts):
    counts = np.array(counts, dtype=np.int64)
    nonzero = counts[counts > 0]
    probs = nonzero / nonzero.sum()
    entropy = -np.sum(probs * np.log(probs + 1e-12))
    normalized_entropy = float(entropy / np.log(len(nonzero))) if len(nonzero) > 1 else 0.0
    return {
        "min_cluster_size": int(nonzero.min()) if len(nonzero) else 0,
        "median_cluster_size": float(np.median(nonzero)) if len(nonzero) else 0.0,
        "max_cluster_size": int(nonzero.max()) if len(nonzero) else 0,
        "normalized_cluster_entropy": normalized_entropy,
        "singleton_clusters": int(np.sum(nonzero == 1)),
    }

def choose_provisional_k(results_df: pd.DataFrame):
    df = results_df.copy()

    def norm_col(col, higher_is_better=True):
        s = df[col].astype(float)
        if s.isna().all():
            return pd.Series(np.nan, index=df.index)
        lo, hi = s.min(skipna=True), s.max(skipna=True)
        if pd.isna(lo) or pd.isna(hi) or hi == lo:
            base = pd.Series(1.0, index=df.index)
        else:
            base = (s - lo) / (hi - lo)
        return base if higher_is_better else 1.0 - base

    score = (
        0.40 * norm_col("mean_pairwise_ari", higher_is_better=True).fillna(0.0) +
        0.20 * norm_col("mean_silhouette", higher_is_better=True).fillna(0.0) +
        0.15 * norm_col("mean_calinski_harabasz", higher_is_better=True).fillna(0.0) +
        0.10 * norm_col("mean_davies_bouldin", higher_is_better=False).fillna(0.0) +
        0.10 * norm_col("mean_normalized_cluster_entropy", higher_is_better=True).fillna(0.0) +
        0.05 * norm_col("mean_min_cluster_size", higher_is_better=True).fillna(0.0)
    )

    penalty = np.where(df["mean_singleton_clusters"].fillna(0) > 0, 0.05, 0.0)
    df["selection_score"] = score - penalty
    best_idx = df["selection_score"].idxmax()
    return df, df.loc[best_idx]

def save_segment_vectors_npz(path: Path, payload: Dict[str, np.ndarray]) -> None:
    np.savez_compressed(
        path,
        vectors=payload["vectors"],
        segment_ids=payload["segment_ids"],
        clip_ids=payload["clip_ids"],
        segment_index=payload["segment_index"],
        start_sec=payload["start_sec"],
        end_sec=payload["end_sec"],
        duration_sec=payload["duration_sec"],
        created_at_utc=np.array([datetime.now(timezone.utc).isoformat()]),
    )

def flatten_token_sequences(jsonl_path: Path) -> pd.DataFrame:
    rows = []
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            record = json.loads(line)
            clip_id = record.get("clip_id")
            metadata = record.get("metadata", {}) or {}
            clip_duration_sec = record.get("clip_duration_sec")
            n_segments = record.get("n_segments")
            segments = record.get("segments", [])
            for seg in segments:
                row = {
                    "clip_id": clip_id,
                    "clip_duration_sec": clip_duration_sec,
                    "n_segments_in_clip": n_segments,
                    "segment_index": int(seg.get("segment_index")),
                    "segment_id": seg.get("segment_id"),
                    "unit_index": int(seg.get("unit_index")),
                    "unit_id": seg.get("unit_id"),
                    "start_sec": float(seg.get("start_sec")),
                    "end_sec": float(seg.get("end_sec")),
                    "duration_sec": float(seg.get("duration_sec")),
                }
                for key, value in metadata.items():
                    row[key] = value
                rows.append(row)
    df = pd.DataFrame(rows)
    if df.empty:
        raise ValueError(f"No rows found in {jsonl_path}")
    return df

def add_global_baselines(df: pd.DataFrame, unit_col: str, value_col: str) -> pd.DataFrame:
    base = df.groupby(value_col).size().rename("global_count").reset_index()
    base["global_proportion"] = base["global_count"] / base["global_count"].sum()

    per_unit = df.groupby([unit_col, value_col]).size().rename("count").reset_index()
    unit_totals = df.groupby(unit_col).size().rename("unit_total").reset_index()

    out = per_unit.merge(unit_totals, on=unit_col, how="left").merge(base, on=value_col, how="left")
    out["unit_proportion"] = out["count"] / out["unit_total"]
    out["lift_vs_global"] = out["unit_proportion"] / out["global_proportion"].replace(0, np.nan)
    return out

def try_load_segment_vectors(npz_path: Path):
    if not npz_path.exists():
        return None, None
    data = np.load(npz_path, allow_pickle=True)
    keys = set(data.files)
    vector_key_candidates = ["vectors", "segment_vectors", "X", "features", "embeddings"]
    id_key_candidates = ["segment_ids", "ids", "segment_id", "row_ids"]
    vector_key = next((k for k in vector_key_candidates if k in keys), None)
    id_key = next((k for k in id_key_candidates if k in keys), None)
    if vector_key is None or id_key is None:
        return None, None
    vectors = data[vector_key]
    segment_ids = data[id_key]
    if len(vectors) != len(segment_ids):
        return None, None
    return vectors, segment_ids

def select_prototypes(unit_df: pd.DataFrame, vectors_lookup: dict | None, top_n: int = 3) -> pd.DataFrame:
    rows = []
    for unit_id, group in unit_df.groupby("unit_id", sort=True):
        group = group.sort_values(["clip_id", "segment_index"]).reset_index(drop=True)
        if vectors_lookup is not None:
            candidate = group[group["segment_id"].isin(vectors_lookup.keys())].copy()
        else:
            candidate = pd.DataFrame()

        if not candidate.empty:
            mat = np.vstack([vectors_lookup[sid] for sid in candidate["segment_id"]])
            centroid = mat.mean(axis=0, keepdims=True)
            distances = np.linalg.norm(mat - centroid, axis=1)
            candidate["prototype_distance"] = distances
            chosen = candidate.nsmallest(min(top_n, len(candidate)), "prototype_distance").copy()
            chosen["prototype_method"] = "nearest_to_unit_centroid"
        else:
            fallback = group.copy()
            median_dur = fallback["duration_sec"].median()
            fallback["prototype_distance"] = (fallback["duration_sec"] - median_dur).abs()
            chosen = fallback.nsmallest(min(top_n, len(fallback)), "prototype_distance").copy()
            chosen["prototype_method"] = "closest_to_unit_median_duration"

        chosen["prototype_rank"] = np.arange(1, len(chosen) + 1)

        keep_cols = [
            "unit_index", "unit_id", "prototype_rank", "prototype_method",
            "segment_id", "clip_id", "segment_index", "start_sec", "end_sec",
            "duration_sec", "main_category_final", "sub_category_final",
            "farm", "barn_zone", "microphone", "mic_placement_context",
            "relative_path", "audio_filename", "file_name"
        ]
        rows.append(chosen[[c for c in keep_cols if c in chosen.columns]])
    if rows:
        return pd.concat(rows, ignore_index=True)
    return pd.DataFrame()

def build_dictionary_outputs(tokens_jsonl: Path, global_freq_csv: Path, segments_csv: Path, segment_vectors_npz: Path, dict_dir: Path) -> Dict[str, Path]:
    token_segments_df = flatten_token_sequences(tokens_jsonl)
    global_freq_df = pd.read_csv(global_freq_csv)
    segments_df = pd.read_csv(segments_csv)

    for col in ["farm", "barn_zone", "microphone", "mic_placement_context", "main_category_final", "sub_category_final"]:
        if col in token_segments_df.columns:
            token_segments_df[col] = token_segments_df[col].map(lambda x: None if pd.isna(x) or str(x).strip() == "" else str(x).strip())

    unit_inventory = (
        token_segments_df.groupby(["unit_index", "unit_id"], as_index=False)
        .agg(
            segment_count=("segment_id", "count"),
            clip_count=("clip_id", "nunique"),
            mean_duration_sec=("duration_sec", "mean"),
            median_duration_sec=("duration_sec", "median"),
            min_duration_sec=("duration_sec", "min"),
            max_duration_sec=("duration_sec", "max"),
            p10_duration_sec=("duration_sec", lambda s: float(np.percentile(s, 10))),
            p90_duration_sec=("duration_sec", lambda s: float(np.percentile(s, 90))),
            mean_segment_index=("segment_index", "mean"),
        )
    )
    unit_inventory["segment_proportion"] = unit_inventory["segment_count"] / unit_inventory["segment_count"].sum()
    unit_inventory["clip_proportion"] = unit_inventory["clip_count"] / token_segments_df["clip_id"].nunique()

    dominant_fields = ["main_category_final", "sub_category_final", "farm", "barn_zone", "microphone", "mic_placement_context"]
    for field in dominant_fields:
        if field in token_segments_df.columns:
            dom = (
                token_segments_df.groupby(["unit_index", field])
                .size()
                .rename("n")
                .reset_index()
                .sort_values(["unit_index", "n", field], ascending=[True, False, True])
                .drop_duplicates(subset=["unit_index"])
                .rename(columns={field: f"dominant_{field}", "n": f"dominant_{field}_count"})
            )
            unit_inventory = unit_inventory.merge(dom, on="unit_index", how="left")

    unit_inventory = unit_inventory.sort_values(["segment_count", "unit_index"], ascending=[False, True]).reset_index(drop=True)
    unit_inventory_path = dict_dir / "unit_inventory.csv"
    unit_inventory.to_csv(unit_inventory_path, index=False)

    category_frames = []
    for field in [f for f in ["main_category_final", "sub_category_final"] if f in token_segments_df.columns]:
        valid = token_segments_df[token_segments_df[field].notna()].copy()
        if valid.empty:
            continue
        assoc = add_global_baselines(valid, unit_col="unit_id", value_col=field)
        assoc.insert(0, "association_field", field)
        assoc = assoc.rename(columns={field: "association_value"})
        assoc["rank_within_unit_by_count"] = assoc.groupby("unit_id")["count"].rank(method="dense", ascending=False).astype(int)
        assoc["rank_within_unit_by_lift"] = assoc.groupby("unit_id")["lift_vs_global"].rank(method="dense", ascending=False).astype(int)
        category_frames.append(assoc)
    unit_category_associations = pd.concat(category_frames, ignore_index=True) if category_frames else pd.DataFrame()
    if not unit_category_associations.empty:
        unit_category_associations = unit_category_associations.sort_values(["unit_id", "association_field", "count", "lift_vs_global"], ascending=[True, True, False, False]).reset_index(drop=True)
    unit_category_path = dict_dir / "unit_category_associations.csv"
    unit_category_associations.to_csv(unit_category_path, index=False)

    context_frames = []
    for field in [f for f in ["farm", "barn_zone", "microphone", "mic_placement_context"] if f in token_segments_df.columns]:
        valid = token_segments_df[token_segments_df[field].notna()].copy()
        if valid.empty:
            continue
        assoc = add_global_baselines(valid, unit_col="unit_id", value_col=field)
        assoc.insert(0, "context_field", field)
        assoc = assoc.rename(columns={field: "context_value"})
        assoc["rank_within_unit_by_count"] = assoc.groupby("unit_id")["count"].rank(method="dense", ascending=False).astype(int)
        assoc["rank_within_unit_by_lift"] = assoc.groupby("unit_id")["lift_vs_global"].rank(method="dense", ascending=False).astype(int)
        context_frames.append(assoc)
    unit_context_associations = pd.concat(context_frames, ignore_index=True) if context_frames else pd.DataFrame()
    if not unit_context_associations.empty:
        unit_context_associations = unit_context_associations.sort_values(["unit_id", "context_field", "count", "lift_vs_global"], ascending=[True, True, False, False]).reset_index(drop=True)
    unit_context_path = dict_dir / "unit_context_associations.csv"
    unit_context_associations.to_csv(unit_context_path, index=False)

    vectors, segment_ids = try_load_segment_vectors(segment_vectors_npz)
    vectors_lookup = None
    if vectors is not None and segment_ids is not None:
        vectors_lookup = {str(sid): np.asarray(vec, dtype=np.float32).reshape(-1) for sid, vec in zip(segment_ids, vectors)}
    unit_prototypes = select_prototypes(token_segments_df, vectors_lookup=vectors_lookup, top_n=3)
    unit_prototypes_path = dict_dir / "unit_prototypes.csv"
    unit_prototypes.to_csv(unit_prototypes_path, index=False)

    summary = {
        "n_units": int(unit_inventory["unit_id"].nunique()),
        "n_segments": int(unit_inventory["segment_count"].sum()),
        "n_clips": int(token_segments_df["clip_id"].nunique()),
        "most_common_unit": unit_inventory.iloc[0]["unit_id"] if not unit_inventory.empty else None,
        "most_common_unit_segments": int(unit_inventory.iloc[0]["segment_count"]) if not unit_inventory.empty else 0,
    }
    summary_path = dict_dir / "dictionary_summary.json"
    with open(summary_path, "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    return {
        "unit_inventory": unit_inventory_path,
        "unit_category_associations": unit_category_path,
        "unit_context_associations": unit_context_path,
        "unit_prototypes": unit_prototypes_path,
        "dictionary_summary_json": summary_path,
    }

def safe_value(x):
    if pd.isna(x):
        return "NA"
    s = str(x).strip()
    return s if s else "NA"

def classify_bias_level(purity, segment_count, dominant_warning_threshold, dominant_high_threshold, min_segments_for_flag):
    if pd.isna(purity):
        return "unknown"
    if segment_count < min_segments_for_flag:
        return "small_sample"
    if purity >= dominant_high_threshold:
        return "high"
    if purity >= dominant_warning_threshold:
        return "moderate"
    return "low"

def build_audit_tables(dict_dir: Path, audits_dir: Path, audits_cfg: Dict[str, Any]) -> Dict[str, Path]:
    audits_dir.mkdir(parents=True, exist_ok=True)
    unit_inventory = pd.read_csv(dict_dir / "unit_inventory.csv")
    context_assoc = pd.read_csv(dict_dir / "unit_context_associations.csv")
    _ = pd.read_csv(dict_dir / "unit_category_associations.csv")

    dominant_warning_threshold = float(audits_cfg.get("dominant_warning_threshold", 0.80))
    dominant_high_threshold = float(audits_cfg.get("dominant_high_threshold", 0.90))
    min_segments_for_flag = int(audits_cfg.get("min_segments_for_flag", 8))

    context_field_to_output = {
        "farm": audits_dir / "unit_farm_bias.csv",
        "barn_zone": audits_dir / "unit_zone_bias.csv",
        "microphone": audits_dir / "unit_microphone_bias.csv",
        "mic_placement_context": audits_dir / "unit_placement_bias.csv",
    }

    dominant_count_cols = {
        "farm": "dominant_farm_count",
        "barn_zone": "dominant_barn_zone_count",
        "microphone": "dominant_microphone_count",
        "mic_placement_context": "dominant_mic_placement_context_count",
    }
    dominant_value_cols = {
        "farm": "dominant_farm",
        "barn_zone": "dominant_barn_zone",
        "microphone": "dominant_microphone",
        "mic_placement_context": "dominant_mic_placement_context",
    }

    def build_bias_table(field_name: str) -> pd.DataFrame:
        field_df = context_assoc.loc[context_assoc["context_field"] == field_name].copy()
        field_df["context_value"] = field_df["context_value"].map(safe_value)

        unit_base = unit_inventory[[
            "unit_id", "unit_index", "segment_count", "clip_count",
            "dominant_main_category_final", "dominant_sub_category_final",
            dominant_value_cols[field_name], dominant_count_cols[field_name],
        ]].copy()

        if field_df.empty:
            out = unit_base.copy()
            out["dominant_context_value"] = out[dominant_value_cols[field_name]].map(safe_value)
            out["dominant_context_count"] = out[dominant_count_cols[field_name]].fillna(0)
            out["dominant_context_proportion"] = out["dominant_context_count"] / out["segment_count"].replace(0, np.nan)
            out["top_context_value"] = np.nan
            out["top_context_count"] = np.nan
            out["top_context_proportion"] = np.nan
            out["top_context_lift_vs_global"] = np.nan
            out["top_context_global_proportion"] = np.nan
            out["top_rank_by_count"] = np.nan
            out["top_rank_by_lift"] = np.nan
            out["n_context_values"] = 0
            out["context_entropy"] = np.nan
            out["normalized_context_entropy"] = np.nan
        else:
            summary = (
                field_df.sort_values(["unit_id", "count", "lift_vs_global"], ascending=[True, False, False])
                .groupby("unit_id", as_index=False)
                .first()
                .rename(columns={
                    "context_value": "top_context_value",
                    "count": "top_context_count",
                    "unit_proportion": "top_context_proportion",
                    "lift_vs_global": "top_context_lift_vs_global",
                    "global_proportion": "top_context_global_proportion",
                    "rank_within_unit_by_count": "top_rank_by_count",
                    "rank_within_unit_by_lift": "top_rank_by_lift",
                })
            )
            out = unit_base.merge(summary[[
                "unit_id", "top_context_value", "top_context_count", "top_context_proportion",
                "top_context_lift_vs_global", "top_context_global_proportion", "top_rank_by_count", "top_rank_by_lift"
            ]], on="unit_id", how="left")

            out["dominant_context_value"] = out[dominant_value_cols[field_name]].map(safe_value)
            out["dominant_context_count"] = out[dominant_count_cols[field_name]]
            out["dominant_context_proportion"] = out["dominant_context_count"] / out["segment_count"].replace(0, np.nan)

            entropy_df = field_df.copy()
            entropy_df["p"] = entropy_df["count"] / entropy_df["unit_total"].replace(0, np.nan)
            entropy = (
                entropy_df.groupby("unit_id")["p"]
                .apply(lambda s: float(-(s * np.log(s + 1e-12)).sum()))
                .reset_index(name="context_entropy")
            )
            nvals = field_df.groupby("unit_id")["context_value"].nunique().reset_index(name="n_context_values")
            out = out.merge(entropy, on="unit_id", how="left").merge(nvals, on="unit_id", how="left")
            out["normalized_context_entropy"] = out["context_entropy"] / np.log(out["n_context_values"].clip(lower=2))
            out["normalized_context_entropy"] = out["normalized_context_entropy"].clip(lower=0, upper=1)

        out["bias_level"] = [
            classify_bias_level(purity, segs, dominant_warning_threshold, dominant_high_threshold, min_segments_for_flag)
            for purity, segs in zip(out["dominant_context_proportion"], out["segment_count"])
        ]
        out["flag_context_locked"] = (
            (out["segment_count"] >= min_segments_for_flag) &
            (out["dominant_context_proportion"] >= dominant_warning_threshold)
        )
        out["flag_highly_locked"] = (
            (out["segment_count"] >= min_segments_for_flag) &
            (out["dominant_context_proportion"] >= dominant_high_threshold)
        )
        out["field_name"] = field_name
        ordered_cols = [
            "field_name", "unit_id", "unit_index", "segment_count", "clip_count",
            "dominant_main_category_final", "dominant_sub_category_final",
            "dominant_context_value", "dominant_context_count", "dominant_context_proportion",
            "top_context_value", "top_context_count", "top_context_proportion",
            "top_context_lift_vs_global", "top_context_global_proportion",
            "n_context_values", "context_entropy", "normalized_context_entropy",
            "bias_level", "flag_context_locked", "flag_highly_locked",
        ]
        return out[ordered_cols].sort_values(
            ["flag_highly_locked", "flag_context_locked", "dominant_context_proportion", "segment_count"],
            ascending=[False, False, False, False]
        ).reset_index(drop=True)

    audit_tables = {}
    for field_name, output_path in context_field_to_output.items():
        bias_table = build_bias_table(field_name)
        audit_tables[field_name] = output_path
        bias_table.to_csv(output_path, index=False)

    summary_rows = []
    for field_name, output_path in audit_tables.items():
        df = pd.read_csv(output_path)
        summary_rows.append({
            "context_field": field_name,
            "n_units": int(len(df)),
            "n_moderate_or_higher": int(df["flag_context_locked"].sum()),
            "n_high": int(df["flag_highly_locked"].sum()),
            "median_dominant_context_proportion": float(df["dominant_context_proportion"].median()) if not df.empty else np.nan,
            "max_dominant_context_proportion": float(df["dominant_context_proportion"].max()) if not df.empty else np.nan,
        })
    summary_df = pd.DataFrame(summary_rows)
    summary_csv = audits_dir / "unit_bias_summary.csv"
    summary_df.to_csv(summary_csv, index=False)

    report_lines = [
        "# Unit Bias Report",
        "",
        "This report is descriptive. It flags units that are strongly concentrated in one recording context.",
        "It does not claim that such units are biologically meaningful or invalid by itself.",
        "",
        "## Audit thresholds",
        f"- Moderate/flag threshold: dominant context proportion >= {dominant_warning_threshold:.2f}",
        f"- High threshold: dominant context proportion >= {dominant_high_threshold:.2f}",
        f"- Minimum segments for flagging: {min_segments_for_flag}",
        "",
        "## Overall summary",
    ]
    for _, row in summary_df.iterrows():
        report_lines.append(
            f"- {row['context_field']}: {int(row['n_moderate_or_higher'])} flagged, {int(row['n_high'])} high, "
            f"median dominant proportion={row['median_dominant_context_proportion']:.3f}, "
            f"max={row['max_dominant_context_proportion']:.3f}"
        )
    write_markdown(audits_dir / "unit_bias_report.md", report_lines)

    return {
        "unit_farm_bias": context_field_to_output["farm"],
        "unit_zone_bias": context_field_to_output["barn_zone"],
        "unit_microphone_bias": context_field_to_output["microphone"],
        "unit_placement_bias": context_field_to_output["mic_placement_context"],
        "unit_bias_summary": summary_csv,
        "unit_bias_report": audits_dir / "unit_bias_report.md",
    }

def infer_token_format(n_units: int, token_cfg: Dict[str, Any]) -> Tuple[str, int | None]:
    prefix = str(token_cfg.get("token_prefix", "u"))
    zero_pad = token_cfg.get("token_zero_pad", None)
    if zero_pad is None:
        zero_pad = len(str(max(1, int(n_units) - 1)))
    zero_pad = max(int(zero_pad), 1)
    return prefix, zero_pad

def format_unit_id(unit_index: int, prefix: str, zero_pad: int | None) -> str:
    unit_index = int(unit_index)
    if zero_pad is None:
        return f"{prefix}{unit_index}"
    return f"{prefix}{unit_index:0{zero_pad}d}"

def build_token_outputs(segments_df: pd.DataFrame, vector_payload: Dict[str, np.ndarray], model_bundle: Dict[str, Any], manifest_df: pd.DataFrame | None, tokens_dir: Path, token_cfg: Dict[str, Any]) -> Dict[str, Path]:
    scaler = model_bundle["scaler"]
    pca = model_bundle["pca"]
    model = model_bundle["model"]

    vector_matrix = vector_payload["vectors"].astype(np.float32)
    vector_segment_ids = vector_payload["segment_ids"].astype(str)
    vector_clip_ids = vector_payload["clip_ids"].astype(str)
    vector_segment_index = vector_payload["segment_index"].astype(np.int32)
    vector_start_sec = vector_payload["start_sec"].astype(np.float32)
    vector_end_sec = vector_payload["end_sec"].astype(np.float32)
    vector_duration_sec = vector_payload["duration_sec"].astype(np.float32)

    X_scaled = scaler.transform(vector_matrix)
    X_cluster = pca.transform(X_scaled) if pca is not None else X_scaled
    unit_index = model.predict(X_cluster).astype(np.int32)

    prefix, zero_pad = infer_token_format(model.n_clusters, token_cfg)
    unit_id = np.array([format_unit_id(u, prefix=prefix, zero_pad=zero_pad) for u in unit_index], dtype="U64")

    vector_df = pd.DataFrame({
        "segment_id": vector_segment_ids,
        "clip_id_from_vectors": vector_clip_ids,
        "segment_index_from_vectors": vector_segment_index,
        "start_sec_from_vectors": vector_start_sec,
        "end_sec_from_vectors": vector_end_sec,
        "duration_sec_from_vectors": vector_duration_sec,
        "unit_index": unit_index,
        "unit_id": unit_id,
    })

    merged_df = segments_df.merge(vector_df, on="segment_id", how="inner")
    if len(merged_df) != len(segments_df):
        raise ValueError(f"Merged token dataframe row mismatch: merged={len(merged_df)} vs segments={len(segments_df)}")

    metadata_cols = []
    if manifest_df is not None and "clip_id" in manifest_df.columns:
        manifest_local = manifest_df.copy()
        manifest_local["clip_id"] = manifest_local["clip_id"].astype(str).map(sanitize_clip_id)
        merged_df = merged_df.merge(manifest_local, on="clip_id", how="left", suffixes=("", "_manifest"))
        metadata_cols = [
            c for c in manifest_local.columns
            if c != "clip_id" and c in merged_df.columns
        ]

    round_time_decimals = int(token_cfg.get("round_time_decimals", 4))
    token_jsonl = tokens_dir / "token_sequences.jsonl"
    token_counts_csv = tokens_dir / "token_counts_by_clip.csv"
    global_freq_csv = tokens_dir / "global_token_frequencies.csv"

    jsonl_records = []
    for clip_id, g in merged_df.groupby("clip_id", sort=False):
        g = g.sort_values(["segment_index", "start_sec", "end_sec", "segment_id"]).reset_index(drop=True)
        record = {
            "clip_id": str(clip_id),
            "clip_duration_sec": float(g["clip_duration_sec"].iloc[0]) if "clip_duration_sec" in g.columns else None,
            "n_segments": int(len(g)),
            "sequence_unit_indices": [int(x) for x in g["unit_index"].tolist()],
            "sequence_unit_ids": [str(x) for x in g["unit_id"].tolist()],
            "segments": [],
        }
        if bool(token_cfg.get("include_manifest_metadata_in_jsonl", True)) and metadata_cols:
            metadata = {}
            first = g.iloc[0]
            for col in metadata_cols:
                val = first[col]
                if pd.isna(val):
                    continue
                metadata[col] = val.item() if hasattr(val, "item") else val
            record["metadata"] = metadata
        for _, row in g.iterrows():
            record["segments"].append({
                "segment_id": str(row["segment_id"]),
                "segment_index": int(row["segment_index"]),
                "unit_index": int(row["unit_index"]),
                "unit_id": str(row["unit_id"]),
                "start_sec": round(float(row["start_sec"]), round_time_decimals),
                "end_sec": round(float(row["end_sec"]), round_time_decimals),
                "duration_sec": round(float(row["duration_sec"]), round_time_decimals),
            })
        jsonl_records.append(record)

    with open(token_jsonl, "w", encoding="utf-8") as f:
        for rec in jsonl_records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

    clip_count_rows = []
    for clip_id, g in merged_df.groupby("clip_id", sort=False):
        g = g.sort_values(["segment_index", "start_sec", "end_sec", "segment_id"]).reset_index(drop=True)
        n_segments_in_clip = int(len(g))
        counts = g.groupby(["unit_index", "unit_id"], as_index=False).size().rename(columns={"size": "count"})
        first_pos = g.groupby(["unit_index", "unit_id"], as_index=False)["segment_index"].min().rename(columns={"segment_index": "first_segment_index"})
        last_pos = g.groupby(["unit_index", "unit_id"], as_index=False)["segment_index"].max().rename(columns={"segment_index": "last_segment_index"})
        count_df = counts.merge(first_pos, on=["unit_index", "unit_id"]).merge(last_pos, on=["unit_index", "unit_id"])
        count_df["clip_id"] = str(clip_id)
        count_df["n_segments_in_clip"] = n_segments_in_clip
        count_df["proportion_of_clip_segments"] = count_df["count"] / max(n_segments_in_clip, 1)
        clip_count_rows.append(count_df)

    token_counts_by_clip_df = pd.concat(clip_count_rows, ignore_index=True)
    token_counts_by_clip_df = token_counts_by_clip_df[[
        "clip_id", "unit_index", "unit_id", "count", "proportion_of_clip_segments", "first_segment_index", "last_segment_index", "n_segments_in_clip"
    ]].sort_values(["clip_id", "unit_index"]).reset_index(drop=True)
    token_counts_by_clip_df.to_csv(token_counts_csv, index=False)

    global_freq_df = (
        merged_df.groupby(["unit_index", "unit_id"], as_index=False)
        .agg(
            segment_count=("segment_id", "count"),
            clip_count=("clip_id", "nunique"),
            mean_duration_sec=("duration_sec", "mean"),
            median_duration_sec=("duration_sec", "median"),
        )
        .sort_values(["segment_count", "unit_index"], ascending=[False, True])
        .reset_index(drop=True)
    )
    global_freq_df["segment_proportion"] = global_freq_df["segment_count"] / global_freq_df["segment_count"].sum()
    global_freq_df["clip_proportion"] = global_freq_df["clip_count"] / merged_df["clip_id"].nunique()
    global_freq_df.to_csv(global_freq_csv, index=False)

    return {
        "token_jsonl": token_jsonl,
        "token_counts_by_clip": token_counts_csv,
        "global_token_frequencies": global_freq_csv,
    }

def compute_sequence_richness(tokens_jsonl: Path) -> Dict[str, float]:
    rows = []
    with open(tokens_jsonl, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rec = json.loads(line)
            seq = [str(x) for x in rec.get("sequence_unit_ids", [])]
            n = len(seq)
            uniq = len(set(seq))
            transitions = sum(1 for a, b in zip(seq[:-1], seq[1:]) if a != b)
            counts = pd.Series(seq).value_counts(normalize=True) if seq else pd.Series(dtype=float)
            entropy = float(-(counts * np.log(counts + 1e-12)).sum()) if not counts.empty else 0.0
            max_entropy = float(np.log(max(len(counts), 1))) if len(counts) > 1 else 0.0
            entropy_norm = safe_div(entropy, max_entropy, default=0.0) if max_entropy > 0 else 0.0
            bigrams = list(zip(seq[:-1], seq[1:]))
            rows.append({
                "clip_id": rec.get("clip_id"),
                "n_segments": n,
                "n_unique_units": uniq,
                "unique_unit_ratio": safe_div(uniq, n, default=0.0),
                "transition_count": transitions,
                "transition_rate": safe_div(transitions, max(n - 1, 1), default=0.0),
                "token_entropy": entropy,
                "token_entropy_normalized": entropy_norm,
                "n_unique_bigrams": len(set(bigrams)),
                "unique_bigram_ratio": safe_div(len(set(bigrams)), max(len(bigrams), 1), default=0.0),
            })
    clip_df = pd.DataFrame(rows)
    if clip_df.empty:
        return {
            "n_token_clips": 0,
            "mean_unique_units_per_clip": np.nan,
            "median_unique_units_per_clip": np.nan,
            "mean_unique_unit_ratio": np.nan,
            "mean_transition_rate": np.nan,
            "mean_token_entropy": np.nan,
            "mean_token_entropy_normalized": np.nan,
            "mean_unique_bigram_ratio": np.nan,
        }
    return {
        "n_token_clips": int(len(clip_df)),
        "mean_unique_units_per_clip": float(clip_df["n_unique_units"].mean()),
        "median_unique_units_per_clip": float(clip_df["n_unique_units"].median()),
        "mean_unique_unit_ratio": float(clip_df["unique_unit_ratio"].mean()),
        "mean_transition_rate": float(clip_df["transition_rate"].mean()),
        "mean_token_entropy": float(clip_df["token_entropy"].mean()),
        "mean_token_entropy_normalized": float(clip_df["token_entropy_normalized"].mean()),
        "mean_unique_bigram_ratio": float(clip_df["unique_bigram_ratio"].mean()),
    }

def compute_confound_summary(audits_dir: Path) -> Dict[str, float]:
    files = {
        "farm": audits_dir / "unit_farm_bias.csv",
        "barn_zone": audits_dir / "unit_zone_bias.csv",
        "microphone": audits_dir / "unit_microphone_bias.csv",
        "mic_placement_context": audits_dir / "unit_placement_bias.csv",
    }
    rows = []
    for field_name, path in files.items():
        df = pd.read_csv(path)
        rows.append({
            "field_name": field_name,
            "n_units": int(len(df)),
            "flagged_units": int(df["flag_context_locked"].sum()),
            "high_units": int(df["flag_highly_locked"].sum()),
            "flagged_unit_pct": float(df["flag_context_locked"].mean() * 100.0) if len(df) else np.nan,
            "high_unit_pct": float(df["flag_highly_locked"].mean() * 100.0) if len(df) else np.nan,
            "median_dominant_context_proportion": float(df["dominant_context_proportion"].median()) if len(df) else np.nan,
            "max_dominant_context_proportion": float(df["dominant_context_proportion"].max()) if len(df) else np.nan,
        })
    summary_df = pd.DataFrame(rows)
    return {
        "confound_fields_evaluated": int(len(summary_df)),
        "mean_flagged_unit_pct": float(summary_df["flagged_unit_pct"].mean()),
        "mean_high_unit_pct": float(summary_df["high_unit_pct"].mean()),
        "mean_median_dominant_context_proportion": float(summary_df["median_dominant_context_proportion"].mean()),
        "max_max_dominant_context_proportion": float(summary_df["max_dominant_context_proportion"].max()),
        "total_flagged_units": int(summary_df["flagged_units"].sum()),
        "total_high_units": int(summary_df["high_units"].sum()),
    }

def candidate_segmentation_metrics(candidate_dir: Path) -> Dict[str, Any]:
    metrics_json = candidate_dir / "candidate_metrics.json"
    if metrics_json.exists():
        return load_json(metrics_json, default={})
    segments_df = pd.read_csv(candidate_dir / "segments.csv")
    summary_df = pd.read_csv(candidate_dir / "segment_summary.csv")
    return {
        "n_clips": int(summary_df["clip_id"].nunique()),
        "n_segments_total": int(len(segments_df)),
        "mean_segments_per_clip": float(summary_df["final_segment_count"].mean()),
        "median_segments_per_clip": float(summary_df["final_segment_count"].median()),
        "single_segment_clip_pct": float((summary_df["final_segment_count"] == 1).mean() * 100.0),
        "tiny_segments_lt_015_pct": float((segments_df["duration_sec"] < 0.15).mean() * 100.0),
        "tiny_segments_lt_020_pct": float((segments_df["duration_sec"] < 0.20).mean() * 100.0),
        "median_segment_duration_sec": float(segments_df["duration_sec"].median()),
    }

def load_baseline_downstream_metrics(output_dir: Path) -> Dict[str, Any]:
    baseline = {}
    kstudy_csv = output_dir / "04_kstudy" / "kstudy_results.csv"
    tokens_jsonl = output_dir / "05_tokens" / "token_sequences.jsonl"
    audits_dir = output_dir / "07_audits"
    if kstudy_csv.exists():
        kstudy_df = pd.read_csv(kstudy_csv)
        if not kstudy_df.empty:
            scored_df, best_row = choose_provisional_k(kstudy_df)
            baseline.update({
                "best_k": int(best_row["k"]),
                "best_mean_pairwise_ari": float(best_row["mean_pairwise_ari"]),
                "best_selection_score": float(best_row["selection_score"]),
            })
    if tokens_jsonl.exists():
        baseline.update(compute_sequence_richness(tokens_jsonl))
    needed_audits = [
        audits_dir / "unit_farm_bias.csv",
        audits_dir / "unit_zone_bias.csv",
        audits_dir / "unit_microphone_bias.csv",
        audits_dir / "unit_placement_bias.csv",
    ]
    if all(p.exists() for p in needed_audits):
        baseline.update(compute_confound_summary(audits_dir))
    return baseline

def normalize_series(series: pd.Series, higher_is_better: bool = True) -> pd.Series:
    s = series.astype(float)
    lo, hi = s.min(skipna=True), s.max(skipna=True)
    if pd.isna(lo) or pd.isna(hi) or hi == lo:
        base = pd.Series(1.0, index=s.index)
    else:
        base = (s - lo) / (hi - lo)
    return base if higher_is_better else 1.0 - base


In [ ]:
# --- Load manifest, sweep plan, and baseline references ---
assert SWEEP_DIR.exists(), f"Missing sweep directory: {SWEEP_DIR}"
assert SWEEP_PLAN_CSV.exists(), f"Missing sweep plan: {SWEEP_PLAN_CSV}. Run 02b first."
assert SWEEP_RESULTS_CSV.exists(), f"Missing sweep results: {SWEEP_RESULTS_CSV}. Run 02b first."
assert BASELINE_REFERENCE_JSON.exists(), f"Missing baseline reference metrics: {BASELINE_REFERENCE_JSON}. Run 02b first."

sweep_plan_df = pd.read_csv(SWEEP_PLAN_CSV)
sweep_results_df = pd.read_csv(SWEEP_RESULTS_CSV)
baseline_seg_metrics = load_json(BASELINE_REFERENCE_JSON, default={})
baseline_downstream_metrics = load_baseline_downstream_metrics(OUTPUT_DIR)
manifest_df = pd.read_csv(MANIFEST_DIR / "manifest.csv") if (MANIFEST_DIR / "manifest.csv").exists() else None
if manifest_df is not None and "clip_id" in manifest_df.columns:
    manifest_df["clip_id"] = manifest_df["clip_id"].astype(str).map(sanitize_clip_id)

print("Sweep candidates found:", len(sweep_plan_df))
print("Manifest available  :", manifest_df is not None)
print("Baseline downstream :", bool(baseline_downstream_metrics))

display(sweep_plan_df[[
    "candidate_id", "candidate_label", "smoothing_window_frames", "change_score_smoothing_frames",
    "peak_prominence", "peak_height_mad_multiplier", "min_peak_distance_sec", "min_segment_duration_sec", "max_segments_per_clip"
]])

print("Baseline segmentation metrics:")
print(json.dumps(baseline_seg_metrics, indent=2)[:2000])
if baseline_downstream_metrics:
    print("Baseline downstream metrics:")
    print(json.dumps(baseline_downstream_metrics, indent=2)[:2000])


In [ ]:
# --- Candidate-local downstream evaluation ---
overwrite_candidate_downstream = bool(CONFIG["segment_sweep"].get("overwrite_candidate_downstream", False))
cl_cfg = CONFIG["clustering"]
audits_cfg = CONFIG["audits"]

evaluation_rows: List[Dict[str, Any]] = []

for candidate in sweep_plan_df.to_dict(orient="records"):
    candidate_id = str(candidate["candidate_id"])
    candidate_dir = SWEEP_DIR / candidate_id
    paths = ensure_candidate_dirs(candidate_dir)

    segments_csv = paths["segments_csv"]
    summary_csv = paths["segment_summary_csv"]
    assert segments_csv.exists(), f"Missing candidate segments: {segments_csv}"
    assert summary_csv.exists(), f"Missing candidate segment summary: {summary_csv}"

    logger.info("Evaluating candidate: %s", candidate_id)
    segments_df = pd.read_csv(segments_csv)
    summary_df = pd.read_csv(summary_csv)

    # 1) Segment vectors
    segment_vectors_npz = paths["segment_vectors_dir"] / "segment_vectors.npz"
    if segment_vectors_npz.exists() and not overwrite_candidate_downstream:
        sv = np.load(segment_vectors_npz, allow_pickle=False)
        vector_payload = {
            "vectors": sv["vectors"].astype(np.float32),
            "segment_ids": sv["segment_ids"].astype(str),
            "clip_ids": sv["clip_ids"].astype(str),
            "segment_index": sv["segment_index"].astype(np.int32),
            "start_sec": sv["start_sec"].astype(np.float32),
            "end_sec": sv["end_sec"].astype(np.float32),
            "duration_sec": sv["duration_sec"].astype(np.float32),
        }
        logger.info("Loaded cached segment vectors for %s", candidate_id)
    else:
        vector_payload = build_segment_vectors(segments_df=segments_df, project_root=PROJECT_ROOT)
        save_segment_vectors_npz(segment_vectors_npz, vector_payload)
        logger.info("Saved segment vectors for %s -> %s", candidate_id, segment_vectors_npz)

    # 2) K-study
    kstudy_dir = paths["kstudy_dir"]
    kstudy_results_csv = kstudy_dir / "kstudy_results.csv"
    kstudy_runs_by_seed_csv = kstudy_dir / "kstudy_runs_by_seed.csv"
    provisional_md = kstudy_dir / "provisional_k_selection.md"
    model_pkl = kstudy_dir / "kmeans_model.pkl"

    if kstudy_results_csv.exists() and model_pkl.exists() and not overwrite_candidate_downstream:
        kstudy_results_df = pd.read_csv(kstudy_results_csv)
        model_bundle = joblib.load(model_pkl)
        logger.info("Loaded cached kstudy/model for %s", candidate_id)
    else:
        vector_matrix = vector_payload["vectors"].astype(np.float32)
        scaler = StandardScaler(with_mean=True, with_std=True)
        X_scaled = scaler.fit_transform(vector_matrix)

        use_pca = bool(cl_cfg.get("use_pca", True))
        pca_components_cfg = int(cl_cfg.get("pca_components", 128))
        pca = None
        X_cluster = X_scaled
        if use_pca:
            n_comp = min(pca_components_cfg, X_scaled.shape[0], X_scaled.shape[1])
            if n_comp >= 2:
                pca = PCA(n_components=n_comp, random_state=int(cl_cfg.get("final_model_seed", 31)))
                X_cluster = pca.fit_transform(X_scaled)

        run_rows = []
        summary_rows = []
        k_values = list(cl_cfg["k_values"])
        seeds = list(cl_cfg["random_seeds"])
        batch_size = int(cl_cfg["batch_size"])
        max_iter = int(cl_cfg["max_iter"])
        n_init = int(cl_cfg["n_init"])
        silhouette_max_samples = int(cl_cfg["silhouette_max_samples"])
        reassignment_ratio = float(cl_cfg.get("reassignment_ratio", 0.01))

        for k in tqdm(k_values, desc=f"K-study {candidate_id}"):
            labels_by_seed = []
            per_seed_rows = []
            for seed in seeds:
                model, labels, counts = run_kmeans_once(
                    X_cluster,
                    k=k,
                    seed=seed,
                    batch_size=batch_size,
                    max_iter=max_iter,
                    n_init=n_init,
                    reassignment_ratio=reassignment_ratio,
                )
                labels_by_seed.append(labels)
                size_info = cluster_size_stats(counts)
                row = {
                    "k": int(k),
                    "seed": int(seed),
                    "inertia": float(model.inertia_),
                    "silhouette": safe_silhouette(X_cluster, labels, max_samples=silhouette_max_samples, seed=seed),
                    "calinski_harabasz": safe_calinski(X_cluster, labels),
                    "davies_bouldin": safe_davies(X_cluster, labels),
                    **size_info,
                }
                per_seed_rows.append(row)
                run_rows.append(row)

            mean_ari, std_ari = pairwise_stability(labels_by_seed)
            seed_df = pd.DataFrame(per_seed_rows)
            summary_rows.append({
                "k": int(k),
                "n_segments": int(X_cluster.shape[0]),
                "n_seeds": int(len(seeds)),
                "mean_pairwise_ari": mean_ari,
                "std_pairwise_ari": std_ari,
                "mean_inertia": float(seed_df["inertia"].mean()),
                "std_inertia": float(seed_df["inertia"].std(ddof=0)),
                "mean_silhouette": float(seed_df["silhouette"].mean()),
                "std_silhouette": float(seed_df["silhouette"].std(ddof=0)),
                "mean_calinski_harabasz": float(seed_df["calinski_harabasz"].mean()),
                "std_calinski_harabasz": float(seed_df["calinski_harabasz"].std(ddof=0)),
                "mean_davies_bouldin": float(seed_df["davies_bouldin"].mean()),
                "std_davies_bouldin": float(seed_df["davies_bouldin"].std(ddof=0)),
                "mean_min_cluster_size": float(seed_df["min_cluster_size"].mean()),
                "mean_median_cluster_size": float(seed_df["median_cluster_size"].mean()),
                "mean_max_cluster_size": float(seed_df["max_cluster_size"].mean()),
                "mean_normalized_cluster_entropy": float(seed_df["normalized_cluster_entropy"].mean()),
                "mean_singleton_clusters": float(seed_df["singleton_clusters"].mean()),
            })

        kstudy_results_df = pd.DataFrame(summary_rows).sort_values("k").reset_index(drop=True)
        kstudy_results_df.to_csv(kstudy_results_csv, index=False)
        pd.DataFrame(run_rows).to_csv(kstudy_runs_by_seed_csv, index=False)

        scored_kstudy_df, best_row = choose_provisional_k(kstudy_results_df)
        best_k = int(best_row["k"])
        final_seed = int(cl_cfg.get("final_model_seed", 31))
        final_model, final_labels, final_counts = run_kmeans_once(
            X_cluster,
            k=best_k,
            seed=final_seed,
            batch_size=batch_size,
            max_iter=max_iter,
            n_init=n_init,
            reassignment_ratio=reassignment_ratio,
        )
        model_bundle = {
            "created_at_utc": datetime.now(timezone.utc).isoformat(),
            "project_root": str(PROJECT_ROOT),
            "selected_k": best_k,
            "final_seed": final_seed,
            "scaler": scaler,
            "pca": pca,
            "model": final_model,
            "segment_vectors_npz": str(segment_vectors_npz),
            "kstudy_results_csv": str(kstudy_results_csv),
            "k_values_tested": k_values,
            "seeds_tested": seeds,
            "n_segments": int(X_cluster.shape[0]),
            "input_dim_raw": int(vector_matrix.shape[1]),
            "input_dim_cluster": int(X_cluster.shape[1]),
            "cluster_size_info": cluster_size_stats(final_counts),
        }
        joblib.dump(model_bundle, model_pkl)

        selection_lines = [
            "# Provisional K Selection",
            "",
            f"- Generated at: {datetime.now(timezone.utc).isoformat()}",
            f"- Candidate: {candidate_id}",
            f"- Number of segments clustered: {X_cluster.shape[0]}",
            f"- Raw segment vector dimension: {vector_matrix.shape[1]}",
            f"- Clustering input dimension: {X_cluster.shape[1]}",
            f"- K values tested: {k_values}",
            f"- Seeds tested: {seeds}",
            "",
            f"- **Provisional K = {best_k}**",
            f"- Mean pairwise ARI: {best_row['mean_pairwise_ari']:.4f}",
            f"- Mean silhouette: {best_row['mean_silhouette']:.4f}",
            f"- Mean Calinski-Harabasz: {best_row['mean_calinski_harabasz']:.4f}",
            f"- Mean Davies-Bouldin: {best_row['mean_davies_bouldin']:.4f}",
            f"- Mean normalized cluster entropy: {best_row['mean_normalized_cluster_entropy']:.4f}",
        ]
        write_markdown(provisional_md, selection_lines)

    scored_kstudy_df, best_row = choose_provisional_k(kstudy_results_df)
    best_k = int(best_row["k"])

    # 3) Token outputs
    token_jsonl = paths["tokens_dir"] / "token_sequences.jsonl"
    token_counts_csv = paths["tokens_dir"] / "token_counts_by_clip.csv"
    global_freq_csv = paths["tokens_dir"] / "global_token_frequencies.csv"
    if token_jsonl.exists() and token_counts_csv.exists() and global_freq_csv.exists() and not overwrite_candidate_downstream:
        token_outputs = {
            "token_jsonl": token_jsonl,
            "token_counts_by_clip": token_counts_csv,
            "global_token_frequencies": global_freq_csv,
        }
        logger.info("Loaded cached token outputs for %s", candidate_id)
    else:
        model_bundle = joblib.load(model_pkl)
        token_outputs = build_token_outputs(
            segments_df=segments_df,
            vector_payload=vector_payload,
            model_bundle=model_bundle,
            manifest_df=manifest_df,
            tokens_dir=paths["tokens_dir"],
            token_cfg=CONFIG["tokens"],
        )
        logger.info("Saved token outputs for %s", candidate_id)

    # 4) Dictionary outputs
    dictionary_summary_json = paths["dict_dir"] / "dictionary_summary.json"
    if dictionary_summary_json.exists() and not overwrite_candidate_downstream:
        dict_outputs = {
            "unit_inventory": paths["dict_dir"] / "unit_inventory.csv",
            "unit_category_associations": paths["dict_dir"] / "unit_category_associations.csv",
            "unit_context_associations": paths["dict_dir"] / "unit_context_associations.csv",
            "unit_prototypes": paths["dict_dir"] / "unit_prototypes.csv",
            "dictionary_summary_json": dictionary_summary_json,
        }
        logger.info("Loaded cached dictionary outputs for %s", candidate_id)
    else:
        dict_outputs = build_dictionary_outputs(
            tokens_jsonl=token_outputs["token_jsonl"],
            global_freq_csv=token_outputs["global_token_frequencies"],
            segments_csv=segments_csv,
            segment_vectors_npz=segment_vectors_npz,
            dict_dir=paths["dict_dir"],
        )
        logger.info("Saved dictionary outputs for %s", candidate_id)

    # 5) Audits
    unit_bias_summary_csv = paths["audits_dir"] / "unit_bias_summary.csv"
    if unit_bias_summary_csv.exists() and not overwrite_candidate_downstream:
        audit_outputs = {
            "unit_farm_bias": paths["audits_dir"] / "unit_farm_bias.csv",
            "unit_zone_bias": paths["audits_dir"] / "unit_zone_bias.csv",
            "unit_microphone_bias": paths["audits_dir"] / "unit_microphone_bias.csv",
            "unit_placement_bias": paths["audits_dir"] / "unit_placement_bias.csv",
            "unit_bias_summary": unit_bias_summary_csv,
            "unit_bias_report": paths["audits_dir"] / "unit_bias_report.md",
        }
        logger.info("Loaded cached audits for %s", candidate_id)
    else:
        audit_outputs = build_audit_tables(
            dict_dir=paths["dict_dir"],
            audits_dir=paths["audits_dir"],
            audits_cfg=audits_cfg,
        )
        logger.info("Saved audits for %s", candidate_id)

    # 6) Candidate score inputs
    seg_metrics = candidate_segmentation_metrics(candidate_dir)
    seq_metrics = compute_sequence_richness(token_outputs["token_jsonl"])
    conf_metrics = compute_confound_summary(paths["audits_dir"])
    dict_summary = load_json(dict_outputs["dictionary_summary_json"], default={})

    eval_row = {
        "candidate_id": candidate_id,
        "candidate_label": candidate.get("candidate_label"),
        "notes": candidate.get("notes"),
        **seg_metrics,
        "best_k": best_k,
        "best_mean_pairwise_ari": float(best_row["mean_pairwise_ari"]),
        "best_mean_silhouette": float(best_row["mean_silhouette"]),
        "best_mean_calinski_harabasz": float(best_row["mean_calinski_harabasz"]),
        "best_mean_davies_bouldin": float(best_row["mean_davies_bouldin"]),
        "best_mean_normalized_cluster_entropy": float(best_row["mean_normalized_cluster_entropy"]),
        "best_mean_singleton_clusters": float(best_row["mean_singleton_clusters"]),
        "best_selection_score": float(best_row["selection_score"]),
        **seq_metrics,
        **conf_metrics,
        "n_units_discovered": int(dict_summary.get("n_units", 0)),
        "n_segments_dictionary": int(dict_summary.get("n_segments", 0)),
        "candidate_root": str(candidate_dir),
        "segment_vectors_npz": str(segment_vectors_npz),
        "kstudy_results_csv": str(kstudy_results_csv),
        "model_pkl": str(model_pkl),
        "token_jsonl": str(token_outputs["token_jsonl"]),
        "dictionary_summary_json": str(dict_outputs["dictionary_summary_json"]),
        "unit_bias_summary_csv": str(audit_outputs["unit_bias_summary"]),
    }

    with open(paths["candidate_eval_metrics_json"], "w", encoding="utf-8") as f:
        json.dump(eval_row, f, indent=2)

    evaluation_rows.append(eval_row)

candidate_eval_df = pd.DataFrame(evaluation_rows)
assert not candidate_eval_df.empty, "No candidate evaluation rows were produced."

display(candidate_eval_df[[
    "candidate_id", "single_segment_clip_pct", "tiny_segments_lt_015_pct", "best_k",
    "best_mean_pairwise_ari", "mean_unique_units_per_clip", "mean_token_entropy_normalized",
    "mean_flagged_unit_pct", "mean_median_dominant_context_proportion"
]].sort_values("candidate_id"))


In [ ]:
# --- Build scorecard, choose winner, and write reports ---
score_df = candidate_eval_df.copy()

baseline_single = safe_float(baseline_seg_metrics.get("single_segment_clip_pct", np.nan), default=np.nan)
baseline_tiny = safe_float(baseline_seg_metrics.get("tiny_segments_lt_015_pct", np.nan), default=np.nan)
baseline_flagged = safe_float(baseline_downstream_metrics.get("mean_flagged_unit_pct", np.nan), default=np.nan)

score_df["single_segment_improvement_pct"] = baseline_single - score_df["single_segment_clip_pct"]
score_df["tiny_fragment_control_value"] = score_df["tiny_segments_lt_015_pct"]
score_df["clustering_quality_value"] = (
    0.60 * score_df["best_mean_pairwise_ari"] +
    0.40 * score_df["best_selection_score"]
)
score_df["sequence_richness_value"] = (
    0.35 * score_df["mean_unique_units_per_clip"] +
    0.35 * score_df["mean_token_entropy_normalized"] +
    0.30 * score_df["mean_transition_rate"]
)
score_df["confound_control_value"] = (
    0.60 * score_df["mean_flagged_unit_pct"] +
    0.40 * (score_df["mean_median_dominant_context_proportion"] * 100.0)
)

score_df["score_segmentation_improvement"] = normalize_series(score_df["single_segment_improvement_pct"], higher_is_better=True)
score_df["score_fragment_control"] = normalize_series(score_df["tiny_fragment_control_value"], higher_is_better=False)
score_df["score_clustering_quality"] = normalize_series(score_df["clustering_quality_value"], higher_is_better=True)
score_df["score_sequence_richness"] = normalize_series(score_df["sequence_richness_value"], higher_is_better=True)
score_df["score_confound_control"] = normalize_series(score_df["confound_control_value"], higher_is_better=False)

w = CONFIG["comparison"]["weights"]
score_df["raw_weighted_score"] = (
    w["segmentation_improvement"] * score_df["score_segmentation_improvement"] +
    w["fragment_control"] * score_df["score_fragment_control"] +
    w["clustering_quality"] * score_df["score_clustering_quality"] +
    w["sequence_richness"] * score_df["score_sequence_richness"] +
    w["confound_control"] * score_df["score_confound_control"]
)

pen_cfg = CONFIG["comparison"]["penalties"]
fragment_abs_limit = float(CONFIG["comparison"].get("fragment_explosion_absolute_pct", 25.0))
fragment_delta_limit = float(CONFIG["comparison"].get("fragment_explosion_delta_pct", 7.0))
confound_delta_limit = float(CONFIG["comparison"].get("confound_worsened_delta_pct", 5.0))

score_df["penalty_no_single_segment_improvement"] = np.where(
    score_df["single_segment_improvement_pct"] <= 0, pen_cfg["no_single_segment_improvement"], 0.0
)
score_df["penalty_fragment_explosion"] = np.where(
    (score_df["tiny_segments_lt_015_pct"] > fragment_abs_limit) |
    ((score_df["tiny_segments_lt_015_pct"] - baseline_tiny) > fragment_delta_limit),
    pen_cfg["fragment_explosion"],
    0.0,
)
if not np.isnan(baseline_flagged):
    score_df["penalty_confound_worsened"] = np.where(
        (score_df["mean_flagged_unit_pct"] - baseline_flagged) > confound_delta_limit,
        pen_cfg["confound_worsened"],
        0.0,
    )
else:
    score_df["penalty_confound_worsened"] = 0.0

score_df["total_penalty"] = (
    score_df["penalty_no_single_segment_improvement"] +
    score_df["penalty_fragment_explosion"] +
    score_df["penalty_confound_worsened"]
)
score_df["final_selection_score"] = score_df["raw_weighted_score"] - score_df["total_penalty"]
score_df = score_df.sort_values(["final_selection_score", "best_mean_pairwise_ari", "single_segment_improvement_pct"], ascending=[False, False, False]).reset_index(drop=True)
score_df.to_csv(COMPARISON_SCORECARD_CSV, index=False)

winner = score_df.iloc[0].to_dict()
runner_up = score_df.iloc[1].to_dict() if len(score_df) > 1 else None

report_lines = []
report_lines.append("# Segmentation Candidate Comparison Report")
report_lines.append("")
report_lines.append(f"- Generated at: {datetime.now(timezone.utc).isoformat()}")
report_lines.append(f"- Candidates evaluated: {len(score_df)}")
report_lines.append(f"- Baseline single-segment clip %: {baseline_single:.3f}")
report_lines.append(f"- Baseline tiny fragment (<0.15s) %: {baseline_tiny:.3f}")
if not np.isnan(baseline_flagged):
    report_lines.append(f"- Baseline mean flagged-unit % across audits: {baseline_flagged:.3f}")
report_lines.append("")
report_lines.append("## Selection score components")
report_lines.append("")
report_lines.append("- Segmentation improvement: lower single-segment dominance than baseline.")
report_lines.append("- Fragment control: avoid explosion of tiny fragments.")
report_lines.append("- Clustering quality: K-study stability and unsupervised separation.")
report_lines.append("- Sequence richness: more informative token sequences for Layer 2.")
report_lines.append("- Confound control: avoid worsening context-locking.")
report_lines.append("")
report_lines.append("## Ranked candidates")
report_lines.append("")
for _, row in score_df.iterrows():
    report_lines.append(
        f"- {row['candidate_id']}: final={row['final_selection_score']:.4f} | "
        f"single-improve={row['single_segment_improvement_pct']:.3f} | "
        f"tiny<0.15={row['tiny_segments_lt_015_pct']:.3f}% | "
        f"ARI={row['best_mean_pairwise_ari']:.4f} | "
        f"seq-rich={row['mean_token_entropy_normalized']:.4f} | "
        f"flagged={row['mean_flagged_unit_pct']:.3f}%"
    )
report_lines.append("")
report_lines.append("## Selected winner")
report_lines.append("")
report_lines.append(f"- Candidate ID: **{winner['candidate_id']}**")
report_lines.append(f"- Label: {winner.get('candidate_label')}")
report_lines.append(f"- Final selection score: {winner['final_selection_score']:.4f}")
report_lines.append(f"- Single-segment improvement vs baseline: {winner['single_segment_improvement_pct']:.3f} percentage points")
report_lines.append(f"- Tiny fragment rate (<0.15s): {winner['tiny_segments_lt_015_pct']:.3f}%")
report_lines.append(f"- Best K: {int(winner['best_k'])}")
report_lines.append(f"- Best mean pairwise ARI: {winner['best_mean_pairwise_ari']:.4f}")
report_lines.append(f"- Mean normalized token entropy: {winner['mean_token_entropy_normalized']:.4f}")
report_lines.append(f"- Mean flagged-unit % across audits: {winner['mean_flagged_unit_pct']:.3f}%")
if runner_up is not None:
    report_lines.append("")
    report_lines.append("## Runner-up")
    report_lines.append("")
    report_lines.append(f"- Candidate ID: {runner_up['candidate_id']}")
    report_lines.append(f"- Final selection score: {runner_up['final_selection_score']:.4f}")

write_markdown(COMPARISON_REPORT_MD, report_lines)

selected_payload = {
    "selected_candidate_id": winner["candidate_id"],
    "selected_candidate_label": winner.get("candidate_label"),
    "selected_at_utc": datetime.now(timezone.utc).isoformat(),
    "selected_candidate_root": str(SWEEP_DIR / winner["candidate_id"]),
    "target_refined_dir": str(REFINED_SEG_DIR),
    "comparison_scorecard_csv": str(COMPARISON_SCORECARD_CSV),
    "comparison_report_md": str(COMPARISON_REPORT_MD),
    "metrics": winner,
}
with open(SELECTED_CANDIDATE_JSON, "w", encoding="utf-8") as f:
    json.dump(selected_payload, f, indent=2)

selected_lines = [
    "# Selected Segmentation Candidate",
    "",
    f"- Selected at: {selected_payload['selected_at_utc']}",
    f"- Candidate ID: **{winner['candidate_id']}**",
    f"- Label: {winner.get('candidate_label')}",
    f"- Candidate root: `{selected_payload['selected_candidate_root']}`",
    f"- Target refined segmentation dir: `{selected_payload['target_refined_dir']}`",
    "",
    "## Why this candidate won",
    "",
    f"- Reduced single-segment dominance by {winner['single_segment_improvement_pct']:.3f} percentage points relative to baseline.",
    f"- Tiny fragment rate (<0.15s) = {winner['tiny_segments_lt_015_pct']:.3f}%.",
    f"- Best K-study ARI = {winner['best_mean_pairwise_ari']:.4f} with provisional K={int(winner['best_k'])}.",
    f"- Mean normalized token entropy = {winner['mean_token_entropy_normalized']:.4f}.",
    f"- Mean flagged-unit % across audits = {winner['mean_flagged_unit_pct']:.3f}%.",
    "",
    "## Required next action",
    "",
    "- Promote this candidate into `output/02_segments_refined/` before the official refined Layer 1 rerun.",
    "- Then point the downstream notebooks to `02_segments_refined` and rerun 03 -> 07.",
]
write_markdown(SELECTED_CANDIDATE_MD, selected_lines)

print("Selected candidate:", winner["candidate_id"])
display(score_df[[
    "candidate_id", "final_selection_score", "single_segment_improvement_pct", "tiny_segments_lt_015_pct",
    "best_k", "best_mean_pairwise_ari", "mean_token_entropy_normalized", "mean_flagged_unit_pct"
]])


In [ ]:
# --- Optional promotion step: copy the selected winner into output/02_segments_refined ---
PROMOTE_SELECTED_CANDIDATE = bool(CONFIG["segment_sweep"].get("promote_selected_candidate", False))
PROMOTE_SELECTED_CANDIDATE = True

selected_info = load_json(SELECTED_CANDIDATE_JSON, default={})
selected_candidate_id = selected_info.get("selected_candidate_id")

if not selected_candidate_id:
    raise ValueError(f"Missing selected candidate metadata: {SELECTED_CANDIDATE_JSON}")

selected_candidate_dir = SWEEP_DIR / selected_candidate_id
selected_segments_csv = selected_candidate_dir / "segments.csv"
selected_summary_csv = selected_candidate_dir / "segment_summary.csv"
selected_config_json = selected_candidate_dir / "candidate_config.json"
selected_metrics_json = selected_candidate_dir / "candidate_metrics.json"
selected_eval_metrics_json = selected_candidate_dir / "candidate_eval_metrics.json"

if PROMOTE_SELECTED_CANDIDATE:
    REFINED_SEG_DIR.mkdir(parents=True, exist_ok=True)
    shutil.copy2(selected_segments_csv, REFINED_SEG_DIR / "segments.csv")
    shutil.copy2(selected_summary_csv, REFINED_SEG_DIR / "segment_summary.csv")
    if selected_config_json.exists():
        shutil.copy2(selected_config_json, REFINED_SEG_DIR / "selected_candidate_config.json")
    if selected_metrics_json.exists():
        shutil.copy2(selected_metrics_json, REFINED_SEG_DIR / "selected_candidate_segmentation_metrics.json")
    if selected_eval_metrics_json.exists():
        shutil.copy2(selected_eval_metrics_json, REFINED_SEG_DIR / "selected_candidate_eval_metrics.json")
    shutil.copy2(SELECTED_CANDIDATE_MD, REFINED_SEG_DIR / "selected_candidate.md")
    shutil.copy2(SELECTED_CANDIDATE_JSON, REFINED_SEG_DIR / "selected_candidate.json")
    print("Promoted selected candidate into:", REFINED_SEG_DIR)
else:
    print("PROMOTE_SELECTED_CANDIDATE is False -> no files copied.")
    print("When ready, set PROMOTE_SELECTED_CANDIDATE = True and run this cell once.")
    print("Target refined dir:", REFINED_SEG_DIR)
